In the following, we provide a code for the 4th step of the proof with genus g=4, considering polynomials of degree 12 and doing the automated eliminations.

In [1]:
import numpy as np
import pandas as pd
from scipy.linalg import companion
from numpy.linalg import matrix_power
from tqdm import tqdm
from sympy import divisors

We first compute the list of polynomials P with degree 12.

In [2]:
degree=12
coeff_candidate=[1,0,0,-1,0,0,0,-1,0,0,-1] #candidate polynomial is x^10-x^7-x^3-1
lmin=max(abs(np.roots(coeff_candidate))) #associated dilatation

We now compute the bounds on the power sums. Since the polynomials we are looking for are skew-reciprocal, we only need to compute half the coefficients and hence half the power sums.

In [3]:
p_up=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_up=np.append(p_up,[int(np.floor(degree/2*(lmin**i+lmin**(-i))))])

print(p_up) #upper bounds on the power sums

[12 12 13 15 17 20]


In [4]:
p_low=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_low=np.append(p_low,[int(np.ceil(min(2-degree,-(degree/2-2)*lmin**i-degree/2*lmin**(-i))))])

print(p_low) #lower bounds on the power sums

[-10 -10 -10 -11 -12 -14]


By Lemma 7, we know the traces of the odd power of the homeomorphisms must be non-negativ. Hence, knowing the power sums for Q, we can redifine the lower bounds for P.

In [5]:
p_low[0]=0
p_low[2]=-3
p_low[4]=0

We now use the Newton's formula to compute the coefficients. We check if the largest root is smaller lmin in absolute value and that it is Perron.

In [6]:
working=[]
for p1 in tqdm(range(p_low[0],p_up[0]+1,1)):
    c1=-p1
    p2_first=(-c1*p1)%2
    for p2 in range(p_up[1]-(p_up[1]-p2_first)%2,p_low[1]-1,-2):
        c2=(p1**2-p2)//2
        p3_first=(-c1*p2-c2*p1)%3
        for p3 in range(p_up[2]-(p_up[2]-p3_first)%3,p_low[2]-1,-3):
            c3=(-c1*p2-c2*p1-p3)//3
            p4_first=(-c1*p3-c2*p2-c3*p1)%4
            for p4 in range(p_up[3]-(p_up[3]-p4_first)%4,p_low[3]-1,-4):
                c4=(-c1*p3-c2*p2-c3*p1-p4)//4
                p5_first=(-c1*p4-c2*p3-c3*p2-c4*p1)%5
                for p5 in range(p_up[4]-(p_up[4]-p5_first)%5,p_low[4]-1,-5):
                    c5=(-c1*p4-c2*p3-c3*p2-c4*p1-p5)//5
                    p6_first=(-c1*p5-c2*p4-c3*p3-c4*p2-c5*p1)%6
                    for p6 in range(p_up[5]-(p_up[5]-p6_first)%6,p_low[5]-1,-6):
                        c6=(-c1*p5-c2*p4-c3*p3-c4*p2-c5*p1-p6)//6
                        coeff=[1,c1,c2,c3,c4,c5,c6,-c5,c4,-c3,c2,-c1,1] #higher power first
                        roots=np.roots(coeff)
                        l=max(roots,key=abs)
                        l2=sorted(abs(roots))[-2]
                        if abs(l)<lmin+0.00001 and l.real>0 and l.imag==0 and np.polyval(coeff,lmin)>-0.00001 and l2<abs(l):
                            coeff.extend([abs(l),p1,p2,p3,p4,p5,p6])
                            working.append(coeff)

100%|██████████| 13/13 [00:12<00:00,  1.02it/s]


In [7]:
polyP=pd.DataFrame(working,columns=['c0','c1','c2','c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','max root','p1','p2','p3','p4','p5','p6'])
print(polyP)

   c0  c1  c2  c3  c4  c5  c6  c7  c8  c9  c10  c11  c12  max root  p1  p2  \
0   1   0  -2  -1   2   2  -2  -2   2   1   -2    0    1  1.173985   0   4   
1   1   0  -2   0  -1   0   4   0  -1   0   -2    0    1  1.000074   0   4   
2   1   0  -2   0   2  -1  -2   1   2   0   -2    0    1  1.159731   0   4   
3   1   0  -2   0   3   0  -4   0   3   0   -2    0    1  1.000000   0   4   
4   1   0  -1  -1   0   1   0  -1   0   1   -1    0    1  1.204425   0   2   
5   1   0  -1  -1   1   1  -2  -1   1   1   -1    0    1  1.173985   0   2   
6   1   0  -1   0   0  -2   0   2   0   0   -1    0    1  1.192766   0   2   
7   1   0   0  -2   0   0  -1   0   0   2    0    0    1  1.173985   0   0   
8   1   0   0  -1  -1   0   0   0  -1   1    0    0    1  1.193859   0   0   
9   1   0   0  -1   0   0  -2   0   0   1    0    0    1  1.173985   0   0   

   p3  p4  p5  p6  
0   3   0   0   7  
1   0  12   0   4  
2   0   0   5   4  
3   0  -4   0   4  
4   3   2   0   5  
5   3  -2   0  11  
6

Some polynomials can be eliminated since they cleared the tests because of the margins we introduced.

In fact the polynomials 4 gives the candidate dilatation.
The polynomials 1,3 are cyclotomic and the polynomials 0,5,6,7,9 are non-primitiv.

We hence get the following list:

In [8]:
polyP=polyP.drop([0,1,3,4,5,6,7,9])
print(polyP)

   c0  c1  c2  c3  c4  c5  c6  c7  c8  c9  c10  c11  c12  max root  p1  p2  \
2   1   0  -2   0   2  -1  -2   1   2   0   -2    0    1  1.159731   0   4   
8   1   0   0  -1  -1   0   0   0  -1   1    0    0    1  1.193859   0   0   

   p3  p4  p5  p6  
2   0   0   5   4  
8   3   4   0   3  


We compute a few power sums.

In [9]:
maxpowersum=37

zeros=np.zeros(len(polyP.index),dtype=np.int64)
for i in range(7,maxpowersum):
    polyP.insert(len(polyP.columns),f'p{i}',zeros)

for idq in polyP.index:
    c=[polyP[f'c{i}'][idq] for i in range(0,13,1)]
    C=companion(c)
    B=companion(c)
    polyP.at[idq,'p1']=np.matrix.trace(C)
    for i in range(2,maxpowersum):
        B=np.matmul(B,C)
        polyP.at[idq,f'p{i}']=np.matrix.trace(B)

We get the polynomials Q we calculated.

In [10]:
polyQ=pd.read_excel('candidates_Q.xlsx',index_col=0,usecols='A:J,L:O',dtype=np.int64)
maxrootQ=pd.read_excel('candidates_Q.xlsx',index_col=0,usecols='A,K')

In [11]:
for idq in polyQ.index:  #compute power sums using compagnon matrix 
    c=[polyQ[f'c{i}'][idq] for i in range(0,9,1)]
    C=companion(c)
    C=np.array(C,dtype=np.int64)
    B=companion(c)
    B=np.array(B,dtype=np.int64)
    polyQ.at[idq,'p1']=np.matrix.trace(C)
    for i in range(2,maxpowersum):
        B=np.matmul(B,C)
        polyQ.at[idq,f'p{i}']=np.matrix.trace(B)

We now eliminate all combinations PQ for which an odd power sum is negative.

In [12]:
working=[]
for idp in polyP.index:
    for idq in polyQ.index:
        if  polyP["p1"][idp]+polyQ["p1"][idq]>=0 and polyP["p3"][idp]+polyQ["p3"][idq]>=0\
        and polyP["p5"][idp]+polyQ["p5"][idq]>=0 and polyP["p7"][idp]+polyQ["p7"][idq]>=0 and polyP["p9"][idp]+polyQ["p9"][idq]>=0\
        and polyP["p11"][idp]+polyQ["p11"][idq]>=0 and polyP["p13"][idp]+polyQ["p13"][idq]>=0 and polyP["p15"][idp]+polyQ["p15"][idq]>=0\
        and polyP["p17"][idp]+polyQ["p17"][idq]>=0 and polyP["p19"][idp]+polyQ["p19"][idq]>=0 and polyP["p21"][idp]+polyQ["p21"][idq]>=0\
        and polyP["p23"][idp]+polyQ["p23"][idq]>=0:
            working.append([idp,idq])

In [13]:
print(len(working))

18


We now apply Lemmas 9,19 and 16 to count singularities.

In [14]:
working2=[]         
for row in working:  
    singularities=0  
    order=[]         # List for the number of points with order exactly 2i+1, with i the index in the list
    for p in range(1,maxpowersum,2):  #Application of Lemma 16
        d_list=divisors(p)            
        points=polyP[f'p{p}'][row[0]]+polyQ[f'p{p}'][row[1]]  
        for d in range(len(d_list)-1):
            points-=order[(d_list[d]-1)//2]  
        order.append(points)   
        if (points/(p))%2==1:  
            singularities+=p     
    for n in [4,6,8,16]:   # Application of Lemma 9 considering f^4,f^6,f^8,f^16
        delta=polyP[f'p{n}'][row[0]]+polyQ[f'p{n}'][row[1]]-polyP[f'p{2*n}'][row[0]]-polyQ[f'p{2*n}'][row[1]] 
        if delta>0:
            singularities+=delta 
    if singularities<=12:  #If there are less than 12 singularities, we push the polynomial in the working list
        working2.append(row)
working=working2  

We do the same test using other powers of f for the Lemma 9.

In [15]:
working2=[]
for row in working:
    singularities=0
    order=[] 
    for p in range(1,maxpowersum,2):
        d_list=divisors(p)
        points=polyP[f'p{p}'][row[0]]+polyQ[f'p{p}'][row[1]]
        for d in range(len(d_list)-1):
            points-=order[(d_list[d]-1)//2]
        order.append(points)
        if (points/(p))%2==1:
            singularities+=p
    for n in [2,4,8,16]:
        delta=polyP[f'p{n}'][row[0]]+polyQ[f'p{n}'][row[1]]-polyP[f'p{2*n}'][row[0]]-polyQ[f'p{2*n}'][row[1]]        
        if delta>0:
            singularities+=delta
    if singularities<=12:
        working2.append(row)
working=working2

We now use Lemma 10 instead of Lemma 9.

In [16]:
deleted=[] 
working2=[]
for row in working:
    singularities=0
    order=[] 
    for p in range(1,maxpowersum,2):
        d_list=divisors(p)
        points=polyP[f'p{p}'][row[0]]+polyQ[f'p{p}'][row[1]]
        for d in range(len(d_list)-1):
            points-=order[(d_list[d]-1)//2]
        order.append(points)
        if (points/(p))%2==1:
            singularities+=p
    for n in [1,3,5,7,9,11,13]: #Application of Lemma 10
        if order[(n-1)//2]==0:  
            delta= 2-polyP[f'p{2*n}'][row[0]]-polyQ[f'p{2*n}'][row[1]]  
        elif order[(n-1)//2]%2==1: 
            delta= 2-polyP[f'p{2*n}'][row[0]]-polyQ[f'p{2*n}'][row[1]]+polyP[f'p{n}'][row[0]]+polyQ[f'p{n}'][row[1]]+2*n  
        else:
            break
        if not(delta<0 or (delta>=0 and singularities+delta<=12)): 
            deleted.append(row)
working = [x for x in working if x not in deleted]

In [17]:
print(len(working))

0
